# Convert 3D STL meshes to SOLWEIG GeoTIFF rasters

This notebook converts:

- `ground_and_water_final.stl` → `dem.tif`
- `building_final.stl` → incorporated into `dsm.tif`
- `vegetation_final.stl` → `cdsm.tif`

The conversion uses downward vertical rays and stores the highest intersected elevation at each raster-cell center, producing a 2.5D representation suitable for SOLWEIG.

Expected folder structure:

```text
/media/harshin/data_drive/solweig/
├── stl/
│   ├── ground_and_water_final.stl
│   ├── building_final.stl
│   └── vegetation_final.stl
└── tif/
```

Important: confirm the CRS before final use. The default below is `EPSG:6346`.

## 1. Install required packages

In [ ]:
%pip install -q "numpy<2" "scipy<1.14" trimesh rasterio rtree matplotlib tqdm

Restart the kernel once after the installation cell if packages were newly installed:

**Kernel → Restart Kernel**

Do not install `pyembree` for this workflow; it can create NumPy dependency conflicts.

## 2. Imports and settings

In [ ]:
from pathlib import Path

import numpy as np
import trimesh
import rasterio
import matplotlib.pyplot as plt

from rasterio.transform import from_origin
from scipy.ndimage import distance_transform_edt
from tqdm import tqdm

In [ ]:
# Main project directories
BASE = Path("/media/harshin/data_drive/solweig")
STL_DIR = BASE / "stl"
TIF_DIR = BASE / "tif"
TIF_DIR.mkdir(parents=True, exist_ok=True)

# Input files
GROUND_STL = STL_DIR / "ground_and_water_final.stl"
BUILDING_STL = STL_DIR / "building_final.stl"
VEGETATION_STL = STL_DIR / "vegetation_final.stl"

# Raster settings
RESOLUTION = 2.0       # metres; start with 2.0, then use 1.0 if needed
CHUNK_ROWS = 20        # reduce if memory is limited
CRS = "EPSG:6346"     # confirm against the original LiDAR/GIS metadata

for path in [GROUND_STL, BUILDING_STL, VEGETATION_STL]:
    print(f"{path.name}: {path.exists()}")

if not all(path.exists() for path in [GROUND_STL, BUILDING_STL, VEGETATION_STL]):
    raise FileNotFoundError("One or more STL input files are missing.")

## 3. Load the STL meshes

In [ ]:
def load_stl_mesh(path):
    obj = trimesh.load(path, process=False)

    if isinstance(obj, trimesh.Scene):
        meshes = [
            geometry
            for geometry in obj.geometry.values()
            if isinstance(geometry, trimesh.Trimesh)
        ]
        if not meshes:
            raise ValueError(f"No triangular meshes found in {path}")
        mesh = trimesh.util.concatenate(meshes)
    else:
        mesh = obj

    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"{path.name} did not load as a triangular mesh")

    mesh.remove_unreferenced_vertices()

    print(f"\nLoaded {path.name}")
    print(f"Vertices: {len(mesh.vertices):,}")
    print(f"Faces:    {len(mesh.faces):,}")
    print("Bounds:")
    print(mesh.bounds)
    print("Extent:")
    print(mesh.extents)

    return mesh


ground_mesh = load_stl_mesh(GROUND_STL)
building_mesh = load_stl_mesh(BUILDING_STL)
vegetation_mesh = load_stl_mesh(VEGETATION_STL)

## 4. Define a common raster grid

In [ ]:
# Use the ground mesh extent for all rasters
xmin = float(ground_mesh.bounds[0, 0])
ymin = float(ground_mesh.bounds[0, 1])
xmax = float(ground_mesh.bounds[1, 0])
ymax = float(ground_mesh.bounds[1, 1])

width = int(np.ceil((xmax - xmin) / RESOLUTION))
height = int(np.ceil((ymax - ymin) / RESOLUTION))

print(f"Extent: {xmin}, {ymin}, {xmax}, {ymax}")
print(f"Raster dimensions: {height} rows × {width} columns")
print(f"Raster cells: {height * width:,}")
print(f"Resolution: {RESOLUTION} m")

## 5. Rasterize each STL using the highest vertical intersection

In [ ]:
def mesh_highest_z_raster(
    mesh,
    xmin,
    ymin,
    xmax,
    ymax,
    resolution,
    chunk_rows=20,
    extra_ray_height=20.0,
    description="Rasterizing",
):
    """Return the highest vertical mesh intersection at every raster-cell center."""

    width = int(np.ceil((xmax - xmin) / resolution))
    height = int(np.ceil((ymax - ymin) / resolution))

    raster_ymax = ymin + height * resolution
    transform = from_origin(
        xmin,
        raster_ymax,
        resolution,
        resolution,
    )

    x_centers = xmin + (np.arange(width) + 0.5) * resolution
    y_centers = raster_ymax - (np.arange(height) + 0.5) * resolution

    output = np.full((height, width), np.nan, dtype=np.float32)
    ray_start_z = float(mesh.bounds[1, 2] + extra_ray_height)

    for row_start in tqdm(
        range(0, height, chunk_rows),
        desc=description,
    ):
        row_end = min(row_start + chunk_rows, height)

        x_grid, y_grid = np.meshgrid(
            x_centers,
            y_centers[row_start:row_end],
        )

        number_of_rays = x_grid.size

        ray_origins = np.column_stack([
            x_grid.ravel(),
            y_grid.ravel(),
            np.full(number_of_rays, ray_start_z),
        ])

        ray_directions = np.zeros_like(ray_origins)
        ray_directions[:, 2] = -1.0

        locations, ray_indices, _ = mesh.ray.intersects_location(
            ray_origins=ray_origins,
            ray_directions=ray_directions,
            multiple_hits=False,
        )

        chunk = np.full(number_of_rays, np.nan, dtype=np.float32)

        if len(locations) > 0:
            chunk[ray_indices] = locations[:, 2].astype(np.float32)

        output[row_start:row_end, :] = chunk.reshape(
            row_end - row_start,
            width,
        )

    return output, transform

In [ ]:
building_z, transform = mesh_highest_z_raster(
    mesh=building_mesh,
    xmin=xmin,
    ymin=ymin,
    xmax=xmax,
    ymax=ymax,
    resolution=RESOLUTION,
    chunk_rows=CHUNK_ROWS,
    description="Building STL",
)

ground_z, _ = mesh_highest_z_raster(
    mesh=ground_mesh,
    xmin=xmin,
    ymin=ymin,
    xmax=xmax,
    ymax=ymax,
    resolution=RESOLUTION,
    chunk_rows=CHUNK_ROWS,
    description="Ground STL",
)

vegetation_z, _ = mesh_highest_z_raster(
    mesh=vegetation_mesh,
    xmin=xmin,
    ymin=ymin,
    xmax=xmax,
    ymax=ymax,
    resolution=RESOLUTION,
    chunk_rows=CHUNK_ROWS,
    description="Vegetation STL",
)

## 6. Fill missing ground cells and construct DEM, DSM, and CDSM

In [ ]:
def fill_nearest(array):
    """Fill NaN cells using the nearest valid raster cell."""
    array = np.asarray(array, dtype=np.float32)
    missing = ~np.isfinite(array)

    if not missing.any():
        return array.copy()

    if missing.all():
        raise ValueError("The raster contains no valid cells.")

    nearest_indices = distance_transform_edt(
        missing,
        return_distances=False,
        return_indices=True,
    )

    return array[tuple(nearest_indices)].astype(np.float32)


missing_count = int(np.isnan(ground_z).sum())
missing_fraction = missing_count / ground_z.size

print(f"Missing ground cells before filling: {missing_count:,}")
print(f"Missing ground fraction: {100 * missing_fraction:.3f}%")

dem = fill_nearest(ground_z)

print("Missing DEM cells after filling:", int(np.isnan(dem).sum()))

In [ ]:
# DSM = ground plus buildings
dsm = dem.copy()

building_mask = np.isfinite(building_z)
dsm[building_mask] = np.maximum(
    dem[building_mask],
    building_z[building_mask],
)

building_height = dsm - dem

# CDSM = vegetation top elevation minus local ground elevation
cdsm = np.zeros_like(dem, dtype=np.float32)

vegetation_mask = np.isfinite(vegetation_z)
cdsm[vegetation_mask] = np.maximum(
    vegetation_z[vegetation_mask] - dem[vegetation_mask],
    0.0,
)

# Remove vegetation where a substantial building occupies the same pixel
cdsm[building_height > 2.0] = 0.0

# Remove very small numerical artifacts
cdsm[cdsm < 0.25] = 0.0

# Keep an explicit clean output array
cdsm_clean = cdsm.copy()

print(f"DEM range:  {np.nanmin(dem):.3f} to {np.nanmax(dem):.3f} m")
print(f"DSM range:  {np.nanmin(dsm):.3f} to {np.nanmax(dsm):.3f} m")
print(f"CDSM range: {np.nanmin(cdsm_clean):.3f} to {np.nanmax(cdsm_clean):.3f} m")
print(f"Maximum building height: {np.nanmax(building_height):.3f} m")

## 7. Inspect canopy-height outliers

In [ ]:
for threshold in [10, 20, 30, 40, 50, 75, 100]:
    print(
        f"CDSM cells above {threshold:3d} m: "
        f"{np.sum(cdsm_clean > threshold):,}"
    )

# Optional cleanup:
# Uncomment only after visually confirming that extreme values are errors.
# cdsm_clean[cdsm_clean > 50.0] = 0.0

## 8. Plot the generated rasters

In [ ]:
def plot_raster(array, title, colorbar_label):
    plt.figure(figsize=(12, 8))
    image = plt.imshow(
        array,
        extent=[xmin, xmax, ymin, ymax],
        origin="upper",
    )
    plt.colorbar(image, label=colorbar_label)
    plt.title(title)
    plt.xlabel("Easting, m")
    plt.ylabel("Northing, m")
    plt.tight_layout()
    plt.show()


plot_raster(dem, "DEM", "Ground elevation, m")
plot_raster(dsm, "DSM: terrain and buildings", "Surface elevation, m")
plot_raster(cdsm_clean, "Vegetation CDSM", "Canopy height above ground, m")

## 9. Write GeoTIFF files

In [ ]:
def write_geotiff(
    output_path,
    array,
    transform,
    crs,
    nodata=-9999.0,
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    data = np.asarray(array, dtype=np.float32).copy()
    data[~np.isfinite(data)] = nodata

    with rasterio.open(
        output_path,
        mode="w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype="float32",
        crs=crs,
        transform=transform,
        nodata=nodata,
        compress="deflate",
        predictor=3,
        tiled=True,
        BIGTIFF="IF_SAFER",
    ) as dataset:
        dataset.write(data, 1)

    print(f"Written: {output_path}")


write_geotiff(TIF_DIR / "dem.tif", dem, transform, CRS)
write_geotiff(TIF_DIR / "dsm.tif", dsm, transform, CRS)
write_geotiff(TIF_DIR / "cdsm.tif", cdsm_clean, transform, CRS)

## 10. Verify the output files

In [ ]:
output_names = ["dem.tif", "dsm.tif", "cdsm.tif"]

for name in output_names:
    path = TIF_DIR / name

    if not path.exists():
        print(f"{name}: MISSING")
        continue

    with rasterio.open(path) as src:
        data = src.read(1, masked=True)

        print(f"\n{name}")
        print("Exists:", path.exists())
        print("Size, bytes:", path.stat().st_size)
        print("Shape:", src.shape)
        print("CRS:", src.crs)
        print("Resolution:", src.res)
        print("Bounds:", src.bounds)
        print("Minimum:", float(data.min()))
        print("Maximum:", float(data.max()))

In [ ]:
with rasterio.open(TIF_DIR / "dem.tif") as dem_src, \
     rasterio.open(TIF_DIR / "dsm.tif") as dsm_src, \
     rasterio.open(TIF_DIR / "cdsm.tif") as cdsm_src:

    print("Same shape:", dem_src.shape == dsm_src.shape == cdsm_src.shape)
    print("Same CRS:", dem_src.crs == dsm_src.crs == cdsm_src.crs)
    print(
        "Same transform:",
        dem_src.transform == dsm_src.transform == cdsm_src.transform,
    )